In [2]:
%pip install pandas requests python-dotenv pyyaml

  Using cached pandas-3.0.6-cp313-cp313-macosx_11_0_arm64.whl.metadata (79 kB)
  Using cached pyyaml-6.0.3-cp313-cp313-macosx_11_0_arm64.whl.metadata (2.4 kB)
Using cached pandas-3.0.6-cp313-cp313-macosx_11_0_arm64.whl (10.1 MB)
Using cached pyyaml-6.0.3-cp313-cp313-macosx_11_0_arm64.whl (173 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 2.5 MB/s  0:00:02 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [pandas]2m2/3 [pandas]

[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import sys
from pathlib import Path

# Find the repo root (the folder that contains src/finsight), wherever this notebook runs from
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "finsight").is_dir())
sys.path.insert(0, str(ROOT / "src"))
import os; os.chdir(ROOT)   # so data/finsight.db and .env are found too

import pandas as pd
from finsight.datastore import api
print("Repo root:", ROOT)

Repo root: /Users/annaphan/Documents/4_Github/AI-Stock-Research-and-Earnings-Assistant


In [4]:
TICKER = "NVDA"
api.get_company(TICKER)
pd.DataFrame(api.list_filings(TICKER, "10-Q", limit=4))
pd.DataFrame(api.get_metric(TICKER, "revenue", "quarter", n=8))

,ticker,metric,fiscal_year,fiscal_period,fiscal_label,period_start,period_end,value,unit,change_vs_prior_quarter_pct,change_vs_prior_year_pct,derived,form,filed,accession,concept,source_url
0,NVDA,revenue,2027,Q2,Q2 FY2027,2026-04-27,2026-07-26,9.622100e+10,USD,17.9,105.9,False,10-Q,2026-08-26,0001045810-26-000075,Revenues,https://www.sec.gov/Archives/edgar/data/104581...
1,NVDA,revenue,2027,Q1,Q1 FY2027,2026-01-26,2026-04-26,8.161500e+10,USD,19.8,85.2,False,10-Q,2026-05-20,0001045810-26-000052,Revenues,https://www.sec.gov/Archives/edgar/data/104581...
2,NVDA,revenue,2026,Q4,Q4 FY2026,2025-10-27,2026-01-25,6.812700e+10,USD,19.5,73.2,True,10-K,2026-02-25,0001045810-26-000021,Revenues,https://www.sec.gov/Archives/edgar/data/104581...
3,NVDA,revenue,2026,Q3,Q3 FY2026,2025-07-28,2025-10-26,5.700600e+10,USD,22.0,62.5,False,10-Q,2025-11-19,0001045810-25-000230,Revenues,https://www.sec.gov/Archives/edgar/data/104581...
4,NVDA,revenue,2026,Q2,Q2 FY2026,2025-04-28,2025-07-27,4.674300e+10,USD,6.1,55.6,False,10-Q,2025-08-27,0001045810-25-000209,Revenues,https://www.sec.gov/Archives/edgar/data/104581...
5,NVDA,revenue,2026,Q1,Q1 FY2026,2025-01-27,2025-04-27,4.406200e+10,USD,12.0,69.2,False,10-Q,2025-05-28,0001045810-25-000116,Revenues,https://www.sec.gov/Archives/edgar/data/104581...
6,NVDA,revenue,2025,Q4,Q4 FY2025,2024-10-28,2025-01-26,3.933100e+10,USD,12.1,77.9,True,10-K,2025-02-26,0001045810-25-000023,Revenues,https://www.sec.gov/Archives/edgar/data/104581...
7,NVDA,revenue,2025,Q3,Q3 FY2025,2024-07-29,2024-10-27,3.508200e+10,USD,16.8,93.6,False,10-Q,2024-11-20,0001045810-24-000316,Revenues,https://www.sec.gov/Archives/edgar/data/104581...


In [5]:
from finsight import config

rows = []
for t in config.load_companies():
    c = api.get_company(t)
    if not c:
        rows.append({"ticker": t, "status": "not loaded"})
        continue
    q10 = api.list_filings(t, "10-Q", limit=1)
    rev = api.get_metric(t, "revenue", "quarter", n=1)
    rows.append({
        "ticker": t,
        "company": c["name"],
        "FY ends": c["fiscal_year_end"],
        "latest 10-Q": q10[0]["fiscal_label"] if q10 else None,
        "revenue ($B)": round(rev[0]["value"] / 1e9, 2) if rev else None,
        "QoQ %": rev[0]["change_vs_prior_quarter_pct"] if rev else None,
        "YoY %": rev[0]["change_vs_prior_year_pct"] if rev else None,
        "filings stored": len(api.list_filings(t, limit=1000)),
    })
pd.DataFrame(rows)

,ticker,company,FY ends,latest 10-Q,revenue ($B),QoQ %,YoY %,filings stored
0,NVDA,NVIDIA CORP,0131,Q2 FY2027,96.22,17.9,105.9,43
1,AAPL,Apple Inc.,0926,Q3 FY2026,109.42,-1.6,16.4,38
2,MSFT,MICROSOFT CORP,0630,Q3 FY2026,90.01,8.6,17.7,38
3,AMD,ADVANCED MICRO DEVICES INC,1226,Q2 FY2026,11.54,12.5,50.1,50
4,AMZN,AMAZON COM INC,1231,Q2 FY2026,200.61,10.5,19.6,43
5,GOOGL,Alphabet Inc.,1231,Q2 FY2026,119.80,9.0,24.2,52
6,META,"Meta Platforms, Inc.",1231,Q2 FY2026,60.80,8.0,28.0,40
7,TSLA,"Tesla, Inc.",1231,Q2 FY2026,28.24,26.1,25.5,45
8,JPM,JPMORGAN CHASE & CO,1231,Q2 FY2026,NaN,NaN,NaN,91
9,INTC,INTEL CORP,1227,Q2 FY2026,16.13,18.8,25.4,65
